# OrganicAI 본실험 — 데이터 수집

담당 시나리오를 **명시/묵시** 두 조건으로 반복 실행해 로그를 Drive 에 쌓는다.
협력 층위(L0~L4) 판정은 수집과 동시에 계산되어 로그에 함께 저장된다.

- **셀 순서 · 담당자별 시나리오 교체 방법 → `docs/MAIN_RUN.md`**
- 채점 기준 → `docs/RUBRIC.md` / 모델 배정 근거 → `docs/MODEL_ASSIGNMENT.md`
- 코드 시험, 새 시나리오 점검은 `OI_Test.ipynb` 에서 한다. **이 노트북은 수집 전용이다.**

## 실행 순서

| 절 | 언제 |
|---|---|
| 0. 설치 | 세션 최초 1회 |
| 1. 세션 시작 | **매 세션 필수** (순서 지킬 것) |
| 2. alpha 대체 | **매 세션 필수** — 빠지면 모든 alpha 호출이 404 |
| 3. 사전 점검 | 첫날 1회 |
| 4. 본수집 | 매 세션 |
| 5. 현황 확인 | 아무 때나 (무료) |

> ⚠️ 런타임을 재시작했으면 0을 건너뛰고 **1부터 다시** 실행한다.
> `git pull` 로 코드를 받은 뒤에도 **반드시 런타임을 재시작**해야 새 코드가 반영된다.

## 0. 설치 (세션 최초 1회)

In [ ]:
!git clone -b yr https://github.com/ewha-oi/OrganicAI.git
%cd OrganicAI
!pip install -r requirements.txt -q

import sys
sys.path.append('src')
print("Python:", sys.version)

## 1. 세션 시작

**런타임 재시작 후에는 여기서부터 실행한다.** 아래 네 셀은 순서를 지킬 것.

In [ ]:
%cd /content/OrganicAI
import sys
sys.path.append('src')
!git log --oneline -1

In [ ]:
from google.colab import userdata


def _secret(name):
    # 등록돼 있지 않으면 userdata.get 이 예외를 던진다. gemini 는 지금 미사용이므로
    # 없다고 세션을 멈출 이유가 없다 -> None 으로 넘기고 표시만 한다.
    try:
        return userdata.get(name)
    except Exception:
        return None


API_KEYS = {
    "gemini": _secret('GEMINI_API_KEY'),   # 미사용 - 없어도 정상 (2절 shim 이 대체)
    "groq":   _secret('GROQ_API_KEY'),     # 필수
}

for k, v in API_KEYS.items():
    mark = 'OK' if v else ('없음 (미사용이라 무방)' if k == 'gemini'
                           else '!! 없음 - Secrets 이름/노트북 액세스 토글 확인')
    print(f"{k:8s} {mark}")

In [ ]:
# 모델 배정. coop_pipeline 을 임포트하는 어떤 셀보다 먼저 실행한다
# (llm.py 가 임포트 시점에 한 번만 읽는다. 값을 바꿨으면 런타임 재시작).
#
# ★ 동결 대상 — 세 사람이 같은 값을 써야 한다. 수집 도중 변경 금지.
#   judge 가 다르면 태깅이 달라져 Kappa 비교가 무효가 된다.
#   배정 근거와 재배정 절차: docs/MODEL_ASSIGNMENT.md

import os
os.environ["COOP_JUDGE_PROVIDER"] = "groq"
os.environ["COOP_JUDGE_MODEL"]    = "qwen/qwen3.6-27b"
os.environ["COOP_ALPHA_MODEL"]    = "openai/gpt-oss-120b"
os.environ["COOP_BETA_MODEL"]     = "openai/gpt-oss-20b"

for k in ("COOP_JUDGE_PROVIDER", "COOP_JUDGE_MODEL",
          "COOP_ALPHA_MODEL", "COOP_BETA_MODEL"):
    print(f"{k:22s} {os.environ[k]}")

In [ ]:
# 담당자가 고치는 셀은 여기 하나뿐이다 — 아래 셀들은 그대로 둔다.
#   교체 방법은 docs/MAIN_RUN.md 「담당자별 시나리오 교체」 참고.
# ============================================================================

# 본인 담당 시나리오. 사람마다 다르다.
# 다른 담당자는 이 목록만 자기 것으로 통째로 바꾼다 (목록이 겹치지 않으면 충돌하지 않는다).
SCENARIOS = [
    "scenarios/A1/A1_complex_gas_alarm.json",
    "scenarios/A1/A1_complex_power_outage.json",
    "scenarios/A1/A1_complex_wifi_outage.json",
    "scenarios/A1/A1_simple_meeting_room.json",
    "scenarios/A1/A1_simple_seminar_hall.json",
    "scenarios/A2/A2_complex_course_slots.json",
    "scenarios/A2/A2_complex_grading_policy.json",
    "scenarios/A2/A2_complex_ta_selection.json",
    "scenarios/A2/A2_simple_mt_venue.json",
    "scenarios/A2/A2_simple_transport.json",
    "scenarios/A4/A4_complex_career_bootcamp.json",
    "scenarios/A4/A4_complex_onboarding.json",
    "scenarios/A4/A4_complex_research_ethics.json",
    "scenarios/A4/A4_simple_festival.json",
    "scenarios/A4/A4_simple_punctuality.json",
    "scenarios/A4/A4_simple_water_save.json",
]

# 반복 수. 시나리오로 분담하므로 rep 은 사람마다 나누지 않는다 — 전원 같은 값을 쓴다.
# 담당 시나리오가 서로 겹치지 않으면 rep 이 같아도 파일명이 겹치지 않는다.
# BATCH 로 끊어 돌린다. 도중에 멈춰도 그때까지가 유효한 데이터다.
#
# 3회로 잡은 이유: judge 의 일일 토큰 한도(TPD 200,000)가 실제 상한을 정한다.
# 담당 16개를 rep 한 바퀴 도는 데만 며칠이 걸린다 — docs/MAIN_RUN.md §9.
MY_REPS = range(1, 4)          # rep 1~3

# ★ 동결 파라미터 — 세 사람이 같은 값. 수집 도중 변경 금지 (노션 동결표).
MAX_TURNS = 10
N_SOLO    = 5                  # 에이전트당 (총 10개)

# 한 번 실행할 때 돌릴 (시나리오 x rep) 수. Groq 무료 티어 일일 한도에 걸리므로
# 하루치씩 끊는다. 16 = 담당 전체를 rep 한 바퀴. None 이면 남은 것을 전부.
BATCH = 16

# 저장 위치 — 팀 공유 폴더 「유기농지능 > data」. 세 사람이 같은 폴더를 써야
# 한 번에 분석할 수 있다. 폴더는 아래 수집 셀이 알아서 만든다.
OUT_DIR = "/content/drive/MyDrive/유기농지능/data"

print(f"담당 시나리오 {len(SCENARIOS)}개 / rep {min(MY_REPS)}~{max(MY_REPS)} "
      f"/ 한 번에 {BATCH}건")
print(f"저장 위치 {OUT_DIR}")

## 2. alpha 모델 대체 (매 세션 필수)

Gemini 접근이 막혀서(전 모델 403/404) alpha 를 Groq 모델로 돌린다.
추론 옵션과 `max_tokens` 를 붙이고 `<think>` 를 지우는 어댑터다. 배경은 `docs/MODEL_ASSIGNMENT.md`.

**런타임을 재시작하면 사라진다. 수집 전에 반드시 다시 실행할 것.**

In [ ]:
# alpha 대체 어댑터 — 런타임을 재시작하면 사라진다. 수집 전에 반드시 실행할 것.
import re, types
from groq import Groq
from coop_pipeline import agents, llm
from coop_pipeline.runner import load_scenario

GROQ_KEY = API_KEYS["groq"]

# 모델 ID 출처는 1절 배정 셀 하나뿐이다. 여기에 하드코딩 금지.
ALPHA_MODEL = llm.MODELS["alpha"]
_THINK = re.compile(r"<think>.*?</think>\s*", re.S)

# 이 모델이 받는 추론 옵션을 런타임에 찾는다 (모델마다 다르고, 틀리면 400).
EXTRA = {}
for cand in ({"reasoning_effort": "low",  "reasoning_format": "hidden"},
             {"reasoning_effort": "none", "reasoning_format": "hidden"},
             {"reasoning_effort": "low"},
             {"reasoning_effort": "none"},
             {"reasoning_format": "hidden"},
             {}):
    try:
        Groq(api_key=GROQ_KEY).chat.completions.create(
            model=ALPHA_MODEL, messages=[{"role": "user", "content": "ping"}],
            max_tokens=64, **cand)
        EXTRA = cand
        break
    except Exception as e:
        print("불가:", cand, "|", str(e)[:90])
print(f"alpha = {ALPHA_MODEL}")
print("사용할 옵션:", EXTRA, "\n")


class _Resp:
    def __init__(self, text): self.text = text


# Groq 는 max_tokens 를 '실제 사용량'이 아니라 '예약량'으로 TPM 예산에 미리 잡는다.
#   요청 비용 = 프롬프트 토큰 + max_tokens
# 대화가 쌓이면 프롬프트가 커지므로, max_tokens 를 크게 두면 뒤쪽 턴에서
# 반드시 413(Request too large)이 난다. 4096 은 6턴짜리에서도 터졌다.
#
# 그렇다고 무작정 줄이면 gpt-oss 는 추론 모델이라 추론 토큰에 다 쓰고 빈 응답이 온다.
# 그래서 넉넉한 값에서 시작해 413 이 나면 절반씩 낮춰 재시도한다.
ALPHA_MAX_TOKENS = 2048       # 시작값
ALPHA_MIN_TOKENS = 768        # 이 아래로는 빈 응답 위험이 커서 낮추지 않는다


class _GroqModel:
    def __init__(self, model_id): self.model_id = model_id

    def generate_content(self, prompt):
        budget = ALPHA_MAX_TOKENS
        while True:
            try:
                r = Groq(api_key=GROQ_KEY).chat.completions.create(
                    model=self.model_id,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=agents.TEMPERATURE,
                    max_tokens=budget,
                    **EXTRA,
                )
                break
            except Exception as e:
                # 413 = 요청 하나가 TPM 한도보다 크다 (기다려도 소용없다. 줄여야 한다).
                # 429 = 분당 한도 초과 (이건 기다리면 된다 -> with_retry 가 처리).
                too_large = getattr(e, "status_code", None) == 413
                if too_large and budget > ALPHA_MIN_TOKENS:
                    budget = max(ALPHA_MIN_TOKENS, budget // 2)
                    print(f"  ! TPM 초과 — max_tokens={budget} 로 낮춰 재시도")
                    continue
                if too_large:
                    raise RuntimeError(
                        f"alpha 요청이 TPM 한도를 넘는다 (max_tokens={budget} 까지 낮췄는데도 실패).\n"
                        f"  프롬프트 자체가 한도보다 크다는 뜻이다 — max_tokens 로는 해결되지 않는다.\n"
                        f"  대화가 길어질수록 더 커지므로 max_turns=10 에서는 반드시 재발한다.\n"
                        f"  TPM 이 더 큰 모델로 alpha 를 재배정하거나 티어를 올려야 한다.\n"
                        f"  현재 모델의 TPM/RPD 확인: docs/MODEL_ASSIGNMENT.md §4 [0]\n"
                        f"  원본: {e}") from e
                raise

        text = _THINK.sub("", r.choices[0].message.content or "").strip()
        if not text:
            raise RuntimeError(
                f"alpha({self.model_id}) 빈 응답 — max_tokens={budget} 를 추론이 다 소진했다. "
                f"ALPHA_MAX_TOKENS 를 올리면 413 위험이 커진다. 현재 EXTRA={EXTRA}")
        return _Resp(text)


_shim = types.SimpleNamespace(configure=lambda **kw: None, GenerativeModel=_GroqModel)
agents._gemini_model = lambda: _shim
agents.RATE_LIMIT_SLEEP = 3.0   # 429 가 뜨면 6.0~10.0 으로 올린다

# 스모크 테스트는 실제 길이의 프롬프트로 한다 (짧은 ping 은 통과해도 실제에서 걸린다).
sc = load_scenario(SCENARIOS[0])
v = sc["task_variants"]
task = v.get("alpha") or v["shared"]
long_prompt = agents.SYSTEM_PROMPT_TEMPLATES["명시"].format(
    name="alpha", partner="beta", task=task
) + "\n\n지금까지의 대화:\n(없음)\n\n너의 다음 발언:"

out = _GroqModel(ALPHA_MODEL).generate_content(long_prompt).text
print(f"길이 {len(out)}자")
print(out[:300])

# TPM 여유 점검 — 대화가 길어질수록 프롬프트가 커져 413 이 난다.
# 1턴짜리 프롬프트가 이미 한도의 상당 부분을 먹으면 10턴은 불가능하다.
raw = Groq(api_key=GROQ_KEY).chat.completions.with_raw_response.create(
    model=ALPHA_MODEL, messages=[{"role": "user", "content": long_prompt}],
    max_tokens=16, **EXTRA)
tpm = int(raw.headers.get("x-ratelimit-limit-tokens", 0) or 0)
used = raw.parse().usage.prompt_tokens

print(f"\nTPM 한도        : {tpm or '?'}")
print(f"1턴 프롬프트    : {used} 토큰")
print(f"요청당 비용     : 프롬프트 + max_tokens({ALPHA_MAX_TOKENS}) = {used + ALPHA_MAX_TOKENS}")
if tpm:
    room = tpm - ALPHA_MAX_TOKENS
    print(f"프롬프트 여유   : {room} 토큰까지 (이걸 넘으면 413)")
    print("→ 대화가 쌓이며 프롬프트가 이 값을 넘으면 자동으로 max_tokens 를 낮춘다.")
    if used + ALPHA_MAX_TOKENS > tpm * 0.5:
        print("!! 1턴부터 한도의 절반을 넘게 쓴다 — max_turns=10 은 중간에 막힐 가능성이 높다.")

## 3. 사전 점검 (첫날 1회)

수집을 시작하기 전에 한 번만 돌린다. **둘째 날부터는 건너뛴다.**
여기서 걸리는 문제는 수집 중에도 똑같이 걸리고, 그때는 2시간짜리 배치 중간에 터진다.

In [ ]:
# [1] 무료 점검 — API 호출 없음
from coop_pipeline.runner import check_scenario_dir
from coop_pipeline.llm import MODELS, judge_provider, judge_model, judge_key_name

!python -m pytest tests/ -q

check_scenario_dir("scenarios")

print(f"\nalpha : {MODELS['alpha']}")
print(f"beta  : {MODELS['beta']}")
print(f"judge : {judge_provider()}:{judge_model()}   (필요한 키: {judge_key_name()})")

In [ ]:
# [2] 태깅 점검 — 채점자가 발화 코드를 제대로 붙이는가. 4/5 이상이면 통과.
#
#     ★ 이 5개는 실행마다 4/5 <-> 3/5 로 흔들린다. 한 번의 3/5 로 판단하지 말 것.
#       - phatic 케이스는 codes 가 정확히 ["phatic"] 하나여야 통과한다 (과잉 태깅
#         탐지용 함정이다). 코드가 하나만 더 붙어도 실패로 세므로 잘 뒤집힌다.
#       - temperature=0 이어도 공유 서빙 환경에서는 실행마다 답이 조금씩 달라진다.
#     판단 기준: 두세 번 돌려 계속 3/5 이하면 그때 판정자를 의심한다.
#     기준을 느슨하게 고쳐서 통과시키지 말 것 — 판정자가 나아진 게 아니라 시험이 쉬워진 것이다.
from coop_pipeline.llm import make_judge, judge_provider, judge_model
from coop_pipeline.tagging import tag_turn

CASES = [
    ("수요일 B실로 하자.",     "좋아, 그렇게 하자.",                                 "phatic"),
    ("수요일에 하는 게 어때?",  "맞네, 나는 A실을 생각했는데 수요일이면 B실이 맞겠다.",  "agree"),
    ("수요일 B실로 하자.",     "좋아. 그런데 예산 확인도 필요해 보여.",                "comp"),
    ("예산은 200이야.",       "응, 200이지.",                                     "phatic"),
    ("회의 준비 시작하자.",    "지금 정할 건 요일이야. 시간은 나중에.",                "lead"),
]

# 배정이 실제로 먹었는지 먼저 확인한다. 여기가 qwen 이 아니면 1절 배정 셀을
# 안 돌렸거나 런타임 재시작 후 다시 안 돌린 것이다 (그 경우 점수를 볼 필요가 없다).
print(f"judge = {judge_provider()}:{judge_model()}\n")

j = make_judge(API_KEYS["groq"])
hit = 0
for prev, cur, want in CASES:
    got = tag_turn(j, [{"turn": 1, "speaker": "alpha", "text": prev}],
                   {"turn": 2, "speaker": "beta", "text": cur})
    ok = (got["codes"] == ["phatic"]) if want == "phatic" else (want in got["codes"])
    hit += ok
    print(f"{'O' if ok else 'X'} 기대={want:6s} 실제={got['codes']} ref={got['ref']}")
    print(f"   근거: {got['evidence']}")      # 왜 그렇게 붙였는지 — 진단의 핵심
print(f"\n{hit}/5")
if hit < 4:
    print("한 번 더 돌려볼 것. 두세 번 연속 3/5 이하일 때만 판정자를 의심한다.")

In [ ]:
# [3] 담당 시나리오 경로 점검 — A1 / A2 / A4 를 하나씩 자동으로 골라 끝까지 돌려본다.
#     A1 은 체크리스트 채점, A2·A4 는 judge 등급으로 코드 경로가 갈린다.
#
#     통과 기준은 하나뿐 — 에러 없이 L0~L4 중 하나가 나오는 것.
#     층위가 낮게 나오는 건 정상이다 (n_solo=2, max_turns=6 축소 설정).
#
#     ★ 로컬 폴더(runs_check)에 쓴다. Drive 의 수집 데이터에 섞이지 않는다.
from coop_pipeline.runner import load_scenario, run_scenario

seen, picks = set(), []
for p in SCENARIOS:
    t = load_scenario(p)["task_type"]
    if t not in seen:
        seen.add(t)
        picks.append((t, p))

print(f"점검 대상 {len(picks)}개: {[t for t, _ in picks]}\n")
for t, p in picks:
    r = run_scenario(p, condition="명시", api_keys=API_KEYS,
                     n_solo=2, max_turns=6, out_dir="runs_check", verbose=False)
    print(f"{t}  {p.split('/')[-1]:36s} -> {r['level']}  "
          f"병목: {r['stopped_at'] or '없음'}")

In [ ]:
# [4] 발언 길이 확인 — max_turns=10 이 토큰 한도 안에 들어오는가. API 호출 0, 무료.
#
#     [3] 은 max_turns=6 으로 돌기 때문에, 통과했다고 10턴이 안전한 것은 아니다.
#     413 이 나는 구간은 9~10턴이다. run_dyad 는 매 턴 '지금까지의 대화 전체'를
#     프롬프트에 넣으므로, 턴이 길면 뒤쪽에서 프롬프트만으로 한도를 넘는다.
#     여기서 실제 발언 길이를 재서 10턴까지 버티는지 미리 계산한다.
from pathlib import Path
import json

CHARS_PER_TOKEN = 1.7      # 한글 실측치
TPM_LIMIT       = 8000     # alpha 모델의 분당 토큰 한도 (2절 shim 출력에서 확인)

logs = sorted(Path("runs_check").glob("*.json"))
if not logs:
    print("runs_check/ 가 비어 있다 — [3] 을 먼저 돌릴 것.")
else:
    worst = 0
    for p in logs:
        log = json.loads(p.read_text(encoding="utf-8"))
        lens = [len(t["text"]) for t in log["turns"]]
        avg = sum(lens) // len(lens)
        worst = max(worst, avg)
        print(f"{p.name:46s} 평균 {avg:5d}자   {lens}")

    # 10턴째 프롬프트 = 과제 지문 + 앞 9턴, 마무리는 10턴 전부를 다시 받는다
    per_turn = worst / CHARS_PER_TOKEN
    turn10   = 252 + 9 * per_turn
    finalize = 252 + 10 * per_turn
    print(f"\n가장 긴 평균: {worst}자 = 약 {per_turn:.0f}토큰/턴")
    print(f"  10턴째 프롬프트 : 약 {turn10:.0f}토큰")
    print(f"  마무리 프롬프트 : 약 {finalize:.0f}토큰   (한도 {TPM_LIMIT})")
    if finalize + 768 < TPM_LIMIT:
        print("\nOK — max_turns=10 이 한도 안에 들어온다. 본수집 진행 가능.")
    else:
        print("\n!! 위험 — 10턴에서 413 이 날 수 있다.")
        print("   발언 길이 규칙이 안 먹은 것이다 (agents.SYSTEM_PROMPT_TEMPLATES 확인).")

## 4. 본수집

아래 셀은 **고치지 않는다.** 바꿀 값은 전부 1절 설정 셀에 있다.

- **이어서 돌리기** — 명시·묵시가 다 저장된 `(시나리오, rep)` 은 건너뛴다. 끊겼으면 그냥 다시 실행.
- **한 건이 실패해도 멈추지 않는다** — 실패 목록을 끝에 찍고, 다시 실행하면 그것만 재시도한다.
- **`BATCH` 만큼만 돌고 멈춘다** — 한도가 회복되면(보통 다음 날) 다시 실행한다.

In [ ]:
# 본수집 루프 — 이 셀은 고치지 않는다. 바꿀 값은 전부 1절 설정 셀에 있다.
import time
from pathlib import Path
from google.colab import drive
from coop_pipeline.runner import load_scenario, run_scenario_both_conditions

drive.mount('/content/drive')
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)


def is_done(sid, rep):
    """명시·묵시 두 조건이 다 저장돼 있어야 완료로 본다."""
    return all((out_dir / f"{sid}_{c}_{rep}.json").exists() for c in ("명시", "묵시"))


# rep 을 바깥 루프에 둔다 -> 담당 전체를 한 바퀴 끝내고 다음 rep 으로 간다.
# 중간에 멈춰도 시나리오별 rep 수가 고르게 유지된다.
jobs = [(p, rep, load_scenario(p)["scenario_id"]) for rep in MY_REPS for p in SCENARIOS]
todo = [j for j in jobs if not is_done(j[2], j[1])]
batch = todo[:BATCH] if BATCH else todo

print(f"저장 위치 : {out_dir}")
print(f"계획 {len(jobs)}건 / 완료 {len(jobs) - len(todo)}건 / 남은 것 {len(todo)}건")
print(f"이번에 돌릴 것 : {len(batch)}건")
print("=" * 78)

t0, failed = time.time(), []
for i, (path, rep, sid) in enumerate(batch, 1):
    try:
        res = run_scenario_both_conditions(
            path, api_keys=API_KEYS, replicate=rep,
            n_solo=N_SOLO, max_turns=MAX_TURNS, out_dir=str(out_dir), verbose=False)
        lv = "  ".join(f"{c}={res[c]['level']}" for c in ("명시", "묵시"))
        print(f"[{i:2d}/{len(batch)}] OK   rep={rep} {sid:34s} {lv}"
              f"   ({(time.time() - t0) / 60:.0f}분)")
    except Exception as e:
        # 한 건이 실패해도 나머지를 계속 돈다. 다시 실행하면 이것만 재시도된다.
        failed.append((sid, rep, str(e)[:150]))
        print(f"[{i:2d}/{len(batch)}] FAIL rep={rep} {sid}: {str(e)[:150]}")

print(f"\n종료 — {(time.time() - t0) / 60:.0f}분, 실패 {len(failed)}건, "
      f"Drive 에 완성 로그 {len(list(out_dir.glob('*.json')))}개")
for sid, rep, msg in failed:
    print(f"  {sid} rep={rep}: {msg}")
if len(todo) > len(batch):
    print(f"\n아직 {len(todo) - len(batch)}건 남았다. "
          f"한도가 회복되면 이 셀을 다시 실행할 것.")

## 5. 현황 확인 (API 호출 0, 무료)

세 사람의 로그가 한 폴더에 모이므로, 누가 돌렸든 여기서 전체가 보인다.

In [ ]:
# 수집 현황 + 전체 판정. API 호출 0, 무료.
from collections import Counter
from pathlib import Path
from google.colab import drive
from coop_pipeline.runner import classify_saved_dir

drive.mount('/content/drive')
out_dir = Path(OUT_DIR)

# 파일명 규칙: {scenario_id}_{condition}_{rep}.json  (raw/ 하위는 원본이라 제외된다)
rows, skipped = [], []
for p in sorted(out_dir.glob("*.json")):
    parts = p.stem.rsplit("_", 2)
    if len(parts) == 3 and parts[2].isdigit() and parts[1] in ("명시", "묵시"):
        rows.append((parts[0], parts[1], int(parts[2])))
    else:
        skipped.append(p.name)

print(f"로그 {len(rows)}개 / 시나리오 {len({s for s, _, _ in rows})}종 "
      f"/ rep {sorted({r for _, _, r in rows})}")
if skipped:
    print(f"(규칙 밖 파일 {len(skipped)}개 무시: {skipped[:3]})")

# 시나리오별 rep 수 — 명시/묵시가 같은 수여야 조건 간 비교가 기운다.
print("\n시나리오별 수집 현황 (명시 / 묵시)")
cnt = Counter((s, c) for s, c, _ in rows)
for sid in sorted({s for s, _, _ in rows}):
    print(f"  {sid:34s} {cnt[(sid, '명시')]:2d} / {cnt[(sid, '묵시')]:2d}")

print()
_ = classify_saved_dir(str(out_dir))

## 부록

In [ ]:
# 최신 코드 받아오기. 받은 뒤에는 반드시 런타임을 재시작하고 1절부터 다시 실행할 것.
!git pull
!git log --oneline -3

In [ ]:
# 임계값 민감도 관찰. ★ 수집이 전부 끝난 뒤에 볼 것.
# v1 사본을 메모리에서만 바꾼다 (로그도 configs/ 도 안 건드린다). API 호출 0, 무료.
#
# 저장되는 것은 로그(발화·태그·점수)이고 판정은 읽을 때 계산되므로,
# 임계값은 수집 시점에 동결할 필요가 없다 — 나중에 몇 번이든 무료로 다시 판정할 수 있다.
# 다만 결과를 보고 기준을 고르면 "협력 효과가 있었다"가 아니라 "기준을 낮췄다"가 된다.
# 확정 절차는 docs/RUBRIC.md 「임계값 캘리브레이션」.
from coop_pipeline.runner import classify_saved_dir
from coop_pipeline import load_thresholds

for gap in (2, 1.5, 1, 0.5):
    print(f"\n### grade_gap_min = {gap}")
    classify_saved_dir(OUT_DIR, dict(load_thresholds("v1"), grade_gap_min=gap))

In [ ]:
# Groq 에서 지금 쓸 수 있는 모델 목록. 모델이 퇴역해 404 가 뜰 때 대체 ID 를 여기서 고른다.
# 고른 뒤 그냥 넣지 말 것 — 재배정 절차는 docs/MODEL_ASSIGNMENT.md
from groq import Groq

for m in sorted(x.id for x in Groq(api_key=API_KEYS["groq"]).models.list().data):
    print("  ", m)